In [4]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import librosa
import os
import glob
import matplotlib.pyplot as plt
import soundfile as sf
from tqdm import tqdm
import random

# Configuration parameters
# Configuration parameters - Updated for 5-second clips
LATENT_DIM = 100
SR = 16000  # Sample rate from MIMII dataset
AUDIO_LENGTH = 5  # seconds (Modified from 10s to 5s)
N_FFT = 1024
HOP_LENGTH = 512
N_MELS = 128
BATCH_SIZE = 8  # Reduced batch size
EPOCHS = 3000
TIME_DIM = 157  # Updated based on 5-second calculation: (5 * SR / HOP_LENGTH) + 1 ≈ 157

def load_and_preprocess_audio(file_path, is_abnormal=False):
    """Enhanced audio preprocessing to better preserve structure - updated for 5s clips"""
    # Load audio file
    audio, _ = librosa.load(file_path, sr=SR, duration=AUDIO_LENGTH)
    
    # Apply pre-emphasis filter to enhance high frequencies
    audio = librosa.effects.preemphasis(audio, coef=0.97)
    
    # Pad if needed
    if len(audio) < SR * AUDIO_LENGTH:
        audio = np.pad(audio, (0, SR * AUDIO_LENGTH - len(audio)))
    elif len(audio) > SR * AUDIO_LENGTH:
        audio = audio[:SR * AUDIO_LENGTH]
    
    # Create mel spectrogram with better parameters
    mel_spec = librosa.feature.melspectrogram(
        y=audio, 
        sr=SR, 
        n_fft=N_FFT, 
        hop_length=HOP_LENGTH, 
        n_mels=N_MELS,
        fmin=50,        # Focus on relevant frequency range
        fmax=8000,      # Focus on relevant frequency range
        power=2.0       # Use power spectrogram
    )
    
    # Convert to log scale with better normalization
    log_mel_spec = librosa.power_to_db(mel_spec, ref=np.max)
    
    # Normalize to [-1, 1] range for tanh activation
    log_mel_spec = 2 * ((log_mel_spec - log_mel_spec.min()) / 
                        (log_mel_spec.max() - log_mel_spec.min())) - 1
    
    # Add channel dimension
    log_mel_spec = np.expand_dims(log_mel_spec, axis=-1)
    
    # Pad to match expected shape if needed
    if log_mel_spec.shape[1] < TIME_DIM:
        pad_width = ((0, 0), (0, TIME_DIM - log_mel_spec.shape[1]), (0, 0))
        log_mel_spec = np.pad(log_mel_spec, pad_width, mode='constant', constant_values=-1)
    elif log_mel_spec.shape[1] > TIME_DIM:
        log_mel_spec = log_mel_spec[:, :TIME_DIM, :]
    
    # Add label for condition (0 for normal, 1 for abnormal)
    condition = 1 if is_abnormal else 0
    
    return log_mel_spec, condition

def process_mimii_dataset(base_dir, machine_type, machine_ids=["00", "02", "04", "06"], samples_per_id=50):
    """Process MIMII dataset files for a specific machine type"""
    specs = []
    conditions = []
    
    print(f"\nProcessing {machine_type} data...")
    machine_dir = os.path.join(base_dir, f"trimmed_{machine_type}")
    
    if not os.path.exists(machine_dir):
        print(f"Error: Directory not found: {machine_dir}")
        return np.array([]), np.array([])
        
    # Process normal samples
    normal_base_dir = os.path.join(machine_dir, "normal")
    if os.path.exists(normal_base_dir):
        # Process each machine ID
        for machine_id in machine_ids:
            normal_id_dir = os.path.join(normal_base_dir, f"id_{machine_id}")
            if os.path.exists(normal_id_dir):
                normal_files = glob.glob(os.path.join(normal_id_dir, "*.wav"))
                
                print(f"Processing {len(normal_files)} normal files for {machine_type} id_{machine_id}")
                # Limit to samples_per_id files per ID for faster processing
                for file_path in tqdm(normal_files[:samples_per_id]):
                    try:
                        spec, condition = load_and_preprocess_audio(file_path, is_abnormal=False)
                        specs.append(spec)
                        conditions.append(condition)
                    except Exception as e:
                        print(f"Error processing {file_path}: {e}")
            else:
                print(f"Directory not found: {normal_id_dir}")
    else:
        print(f"Normal directory not found: {normal_base_dir}")
        
    # Process abnormal samples
    abnormal_base_dir = os.path.join(machine_dir, "abnormal")
    if os.path.exists(abnormal_base_dir):
        # Process each machine ID
        for machine_id in machine_ids:
            abnormal_id_dir = os.path.join(abnormal_base_dir, f"id_{machine_id}")
            if os.path.exists(abnormal_id_dir):
                abnormal_files = glob.glob(os.path.join(abnormal_id_dir, "*.wav"))
                
                print(f"Processing {len(abnormal_files)} abnormal files for {machine_type} id_{machine_id}")
                # Limit to samples_per_id files per ID for faster processing
                for file_path in tqdm(abnormal_files[:samples_per_id]):
                    try:
                        spec, condition = load_and_preprocess_audio(file_path, is_abnormal=True)
                        specs.append(spec)
                        conditions.append(condition)
                    except Exception as e:
                        print(f"Error processing {file_path}: {e}")
            else:
                print(f"Directory not found: {abnormal_id_dir}")
    else:
        print(f"Abnormal directory not found: {abnormal_base_dir}")
    
    # If no files were found, create some dummy data to avoid errors
    if len(specs) == 0:
        print("WARNING: No files were processed. Creating dummy data for debugging.")
        return np.zeros((10, N_MELS, TIME_DIM, 1)), np.zeros(10)
    
    # Convert to numpy arrays
    specs = np.array(specs)
    conditions = np.array(conditions)
    
    print(f"Dataset shape: {specs.shape}, Conditions shape: {conditions.shape}")
    print(f"Normal samples: {np.sum(conditions == 0)}, Abnormal samples: {np.sum(conditions == 1)}")
    
    return specs, conditions

# Custom Resizing Layer to ensure exact output dimensions
class ExactOutputSize(layers.Layer):
    def __init__(self, target_height, target_width, **kwargs):
        super(ExactOutputSize, self).__init__(**kwargs)
        self.target_height = target_height
        self.target_width = target_width
        
    def build(self, input_shape):
        super(ExactOutputSize, self).build(input_shape)
        
    def call(self, inputs):
        return tf.image.resize(inputs, [self.target_height, self.target_width])
    
    def compute_output_shape(self, input_shape):
        return (input_shape[0], self.target_height, self.target_width, input_shape[3])
        
    def get_config(self):
        config = super(ExactOutputSize, self).get_config()
        config.update({
            'target_height': self.target_height,
            'target_width': self.target_width
        })
        return config

def self_attention_layer(x):
    """Implement self-attention mechanism for spectrograms"""
    batch_size, height, width, channels = x.shape
    
    # Create query, key, value projections
    f = layers.Conv2D(channels // 8, kernel_size=1, strides=1, padding='same')(x)
    g = layers.Conv2D(channels // 8, kernel_size=1, strides=1, padding='same')(x)
    h = layers.Conv2D(channels, kernel_size=1, strides=1, padding='same')(x)
    
    # Reshape for matrix multiplication
    f_flat = layers.Reshape((height * width, channels // 8))(f)
    g_flat = layers.Reshape((height * width, channels // 8))(g)
    h_flat = layers.Reshape((height * width, channels))(h)
    
    # Compute attention map
    s = layers.Lambda(lambda x: tf.matmul(x[0], tf.transpose(x[1], [0, 2, 1])))([f_flat, g_flat])
    beta = layers.Softmax(axis=-1)(s)
    
    # Apply attention
    o = layers.Lambda(lambda x: tf.matmul(x[0], x[1]))([beta, h_flat])
    o = layers.Reshape((height, width, channels))(o)
    
    # Scale and add with input (residual connection)
    gamma = tf.Variable(0., trainable=True)
    x = layers.add([x, gamma * o])
    
    return x

def build_generator(latent_dim):
    """Build improved generator network with better structure preservation for audio spectrograms"""
    # Input for latent space
    noise = layers.Input(shape=(latent_dim,), name="noise_input")
    
    # Input for conditions (normal/abnormal)
    condition = layers.Input(shape=(1,), name="condition_input")
    
    # Embed the condition with more information
    condition_embedding = layers.Embedding(2, 64)(condition)
    condition_embedding = layers.Flatten()(condition_embedding)
    condition_embedding = layers.Dense(128, activation='relu')(condition_embedding)
    
    # Concatenate noise and condition
    combined_input = layers.Concatenate()([noise, condition_embedding])
    
    # Initial dense layers
    x = layers.Dense(512)(combined_input)
    x = layers.LeakyReLU(alpha=0.2)(x)
    x = layers.BatchNormalization(momentum=0.8)(x)
    
    x = layers.Dense(1024)(x)
    x = layers.LeakyReLU(alpha=0.2)(x)
    x = layers.BatchNormalization(momentum=0.8)(x)
    
    # Calculate dimensions for reshaping
    target_height = N_MELS // 16  # 8 for 128 mel bands
    target_width = TIME_DIM // 16  # For time dimension
    
    x = layers.Dense(target_height * target_width * 256)(x)
    x = layers.LeakyReLU(alpha=0.2)(x)
    x = layers.BatchNormalization(momentum=0.8)(x)
    x = layers.Reshape((target_height, target_width, 256))(x)
    
    # First upsampling block with skip connection
    x = layers.Conv2DTranspose(128, (5, 9), strides=(2, 2), padding='same')(x)
    x = layers.LeakyReLU(alpha=0.2)(x)
    x = layers.BatchNormalization(momentum=0.8)(x)
    skip1 = x
    
    # Second upsampling block with skip connection
    x = layers.Conv2DTranspose(64, (5, 9), strides=(2, 2), padding='same')(x)
    x = layers.LeakyReLU(alpha=0.2)(x)
    x = layers.BatchNormalization(momentum=0.8)(x)
    skip2 = x
    
    # Third upsampling block with skip connection
    x = layers.Conv2DTranspose(32, (5, 7), strides=(2, 2), padding='same')(x)
    x = layers.LeakyReLU(alpha=0.2)(x)
    x = layers.BatchNormalization(momentum=0.8)(x)
    
    # Add skip connection to help preserve structure
    skip2_up = layers.Conv2DTranspose(32, (1, 1), strides=(2, 2), padding='same')(skip2)
    x = layers.add([x, skip2_up])
    
    # Fourth upsampling block
    x = layers.Conv2DTranspose(16, (5, 5), strides=(2, 2), padding='same')(x)
    x = layers.LeakyReLU(alpha=0.2)(x)
    x = layers.BatchNormalization(momentum=0.8)(x)
    
    # Add additional skip connection for better structure preservation
    skip1_up = layers.Conv2DTranspose(16, (1, 1), strides=(8, 8), padding='same')(skip1)
    x = layers.add([x, skip1_up])
    
    # Resize to exact dimensions
    x = ExactOutputSize(N_MELS, TIME_DIM)(x)
    
    # Add frequency-aware convolution - taller kernel emphasizes frequency patterns
    x = layers.Conv2D(8, (9, 3), padding='same')(x)
    x = layers.LeakyReLU(alpha=0.2)(x)
    
    # Apply low-pass filtering to match the real spectrograms' smoothness
    # This is critical - real spectrograms don't have the high-frequency noise seen in generations
    x = layers.AveragePooling2D(pool_size=(2, 2), strides=(1, 1), padding='same')(x)
    x = layers.Conv2D(8, (3, 3), padding='same')(x)
    x = layers.LeakyReLU(alpha=0.2)(x)
    
    # Add time-aware convolution - wider kernel captures temporal patterns
    x = layers.Conv2D(4, (3, 9), padding='same')(x)
    x = layers.LeakyReLU(alpha=0.2)(x)
    
    # Final processing to generate the clean, structured output
    x = layers.Conv2D(2, (5, 5), padding='same')(x)
    x = layers.LeakyReLU(alpha=0.2)(x)
    
    # Output layer with tanh activation to match normalized range
    output = layers.Conv2D(1, (3, 3), activation='tanh', padding='same')(x)
    
    # Apply spectral structure constraint to match real spectrograms
    # This helps ensure the output follows the bluish-green pattern seen in real specs
    output = layers.Lambda(lambda x: x * 0.5)(output)  # Scale down to match real spectrogram range
    
    model = models.Model([noise, condition], output, name="generator")
    return model

# Discriminator network
def build_discriminator():
    """Build improved discriminator network for audio spectrograms"""
    # Input for spectrogram with explicit shape
    input_spec = layers.Input(shape=(N_MELS, TIME_DIM, 1), name="spec_input")
    
    # Input for conditions (normal/abnormal)
    condition = layers.Input(shape=(1,), name="condition_input")
    
    # Embedding layer for condition - reshape to match input spec dimensions
    embedding_dim = 64
    condition_embedding = layers.Embedding(2, embedding_dim)(condition)
    condition_embedding = layers.Dense(N_MELS * TIME_DIM)(condition_embedding)
    condition_embedding = layers.Reshape((N_MELS, TIME_DIM, 1))(condition_embedding)
    
    # Concatenate inputs along the channel dimension
    combined_input = layers.Concatenate(axis=-1)([input_spec, condition_embedding])
    
    # Convolutional layers with attention to time-frequency patterns
    # Using larger kernels for the time dimension to capture temporal patterns
    x = layers.Conv2D(32, (5, 9), strides=(2, 2), padding='same')(combined_input)
    x = layers.LeakyReLU(alpha=0.2)(x)
    x = layers.Dropout(0.3)(x)
    
    x = layers.Conv2D(64, (5, 9), strides=(2, 2), padding='same')(x)
    x = layers.LeakyReLU(alpha=0.2)(x)
    x = layers.Dropout(0.3)(x)
    
    x = layers.Conv2D(128, (5, 7), strides=(2, 2), padding='same')(x)
    x = layers.LeakyReLU(alpha=0.2)(x)
    x = layers.Dropout(0.3)(x)
    
    x = layers.Conv2D(256, (5, 5), strides=(2, 2), padding='same')(x)
    x = layers.LeakyReLU(alpha=0.2)(x)
    x = layers.Dropout(0.3)(x)
    
    # Attention mechanism to focus on important features
    # Self-attention layer to capture long-range dependencies
    x = self_attention_layer(x)
    
    # Flatten and dense layers
    x = layers.Flatten()(x)
    x = layers.Dense(512)(x)
    x = layers.LeakyReLU(alpha=0.2)(x)
    
    # Output layer
    output = layers.Dense(1, activation='sigmoid')(x)
    
    # Model
    model = models.Model([input_spec, condition], output, name="discriminator")
    
    return model

# GAN model
def build_gan(generator, discriminator):
    """Build combined GAN model"""
    # Freeze discriminator during generator training
    discriminator.trainable = False
    
    # GAN input
    noise = layers.Input(shape=(LATENT_DIM,), name="gan_noise_input")
    condition = layers.Input(shape=(1,), name="gan_condition_input")
    
    # Generator output
    generated_spec = generator([noise, condition])
    
    # Discriminator output
    validity = discriminator([generated_spec, condition])
    
    # Combined model
    model = models.Model([noise, condition], validity, name="gan")
    
    return model

def spectrogram_to_audio(spectrogram):
    """Enhanced conversion from spectrogram to audio with better matching to real spectrograms"""
    # Remove extra dimensions
    spectrogram = spectrogram.squeeze()
    
    # Apply smoothing to reduce noise and make more like real spectrograms
    from scipy.ndimage import gaussian_filter
    spectrogram = gaussian_filter(spectrogram, sigma=0.5)
    
    # Apply soft thresholding to reduce extreme values (real spectrograms don't have many extremes)
    spectrogram = np.tanh(spectrogram)
    
    # Rescale from [-1, 1] to [0, 1] range
    spectrogram = (spectrogram + 1) / 2
    
    # Convert to dB scale with range matching real spectrograms
    # Real spectrograms are mostly in the blue-green range (-0.50 to 0.25)
    db_spectrogram = spectrogram * 50 - 65  # Map to dB range that emphasizes mid/low values
    
    # Apply frequency-dependent scaling to emphasize lower frequencies where pump energy is
    freq_scaling = np.linspace(1.2, 0.6, N_MELS).reshape(-1, 1)
    db_spectrogram = db_spectrogram * freq_scaling
    
    # Convert from dB back to power
    S = librosa.db_to_power(db_spectrogram)
    
    # Apply a low-pass filter to remove high-frequency noise
    freq_window = np.hamming(N_MELS).reshape(-1, 1)
    S = S * freq_window
    
    # Mel spectrogram to audio using Griffin-Lim algorithm
    y = librosa.feature.inverse.mel_to_audio(
        S, 
        sr=SR, 
        n_fft=N_FFT, 
        hop_length=HOP_LENGTH,
        fmin=50,
        fmax=8000,
        power=2.0,
        n_iter=64
    )
    
    return y

def train_gan(generator, discriminator, gan, specs, conditions, epochs, batch_size, machine_type):
    """Improved GAN training function with better spectral matching"""
    
    # Create folder for samples
    output_dir = f"generated_samples_{machine_type}"
    os.makedirs(output_dir, exist_ok=True)
    
    # Save training info
    with open(f"{output_dir}/training_info.txt", "w") as f:
        f.write(f"Machine Type: {machine_type}\n")
        f.write(f"Dataset Size: {specs.shape[0]} samples\n")
        f.write(f"Normal Samples: {np.sum(conditions == 0)}\n")
        f.write(f"Abnormal Samples: {np.sum(conditions == 1)}\n")
        f.write(f"Training Epochs: {epochs}\n")
        f.write(f"Batch Size: {batch_size}\n")
        f.write(f"Sample Rate: {SR}\n")
        f.write(f"Audio Length: {AUDIO_LENGTH}s\n")
        f.write(f"Latent Dimension: {LATENT_DIM}\n")
    
    # Arrays to store losses
    d_losses = []
    g_losses = []
    feature_losses = []
    spectral_losses = []
    
    # Feature extractor from discriminator
    feature_extractor = models.Model(
        inputs=discriminator.input,
        outputs=discriminator.layers[-2].output
    )
    
    # Get statistical properties of real spectrograms for improved matching
    sample_size = min(500, specs.shape[0])
    reference_specs = specs[:sample_size]
    reference_mean = np.mean(reference_specs, axis=0)
    reference_std = np.std(reference_specs, axis=0)
    
    # Get color distribution of real spectrograms (histogram)
    # This helps match the color characteristics of real spectrograms
    real_hist, _ = np.histogram(reference_specs.flatten(), bins=50, range=(-1, 1), density=True)
    
    # Set target ratio for discriminator/generator training
    # Starting with more discriminator training to ensure it can distinguish real/fake
    disc_train_ratio = 2
    
    # Training loop with adaptive balance and spectral constraints
    for epoch in range(epochs):
        # Update training ratio based on loss balance (helps stabilize training)
        if epoch > 200 and epoch % 100 == 0:
            if np.mean(d_losses[-20:]) < 0.4:
                disc_train_ratio = max(1, disc_train_ratio - 1)  # Reduce discriminator training
            elif np.mean(d_losses[-20:]) > 0.7:
                disc_train_ratio = min(5, disc_train_ratio + 1)  # Increase discriminator training
        
        # Train discriminator for multiple steps to maintain balance
        d_loss_epoch = 0
        for _ in range(disc_train_ratio):
            # Get real samples
            idx = np.random.randint(0, specs.shape[0], batch_size // 2)
            real_specs = specs[idx]
            real_conditions = conditions[idx].reshape(-1, 1)
            
            # Generate fake samples
            noise = np.random.normal(0, 1, (batch_size // 2, LATENT_DIM))
            fake_conditions = real_conditions.copy()
            gen_specs = generator.predict([noise, fake_conditions])
            
            # Add noise to labels for label smoothing (stabilizes training)
            real_labels = np.ones((batch_size // 2, 1)) * 0.9
            fake_labels = np.zeros((batch_size // 2, 1)) + 0.1
            
            # Train discriminator
            d_loss_real = discriminator.train_on_batch([real_specs, real_conditions], real_labels)
            d_loss_fake = discriminator.train_on_batch([gen_specs, fake_conditions], fake_labels)
            
            if isinstance(d_loss_real, list):
                d_loss = 0.5 * (d_loss_real[0] + d_loss_fake[0])
            else:
                d_loss = 0.5 * (d_loss_real + d_loss_fake)
                
            d_loss_epoch += d_loss
        
        d_loss = d_loss_epoch / disc_train_ratio
        
        # Train generator
        noise = np.random.normal(0, 1, (batch_size, LATENT_DIM))
        gen_conditions = np.concatenate([
            np.zeros((batch_size // 2, 1)),   # Normal
            np.ones((batch_size // 2, 1))     # Abnormal
        ])
        
        # Generate fake spectrograms
        gen_specs = generator.predict([noise, gen_conditions])
        
        # Extract features for feature matching
        real_idx = np.random.randint(0, specs.shape[0], batch_size)
        real_specs_matched = specs[real_idx]
        real_conditions_matched = conditions[real_idx].reshape(-1, 1)
        
        real_features = feature_extractor.predict([real_specs_matched, real_conditions_matched])
        fake_features = feature_extractor.predict([gen_specs, gen_conditions])
        
        # Calculate feature matching loss
        feature_loss = np.mean(np.abs(real_features - fake_features))
        feature_losses.append(feature_loss)
        
        # Calculate spectral statistics matching loss
        gen_specs_mean = np.mean(gen_specs, axis=0)
        gen_specs_std = np.std(gen_specs, axis=0)
        
        stats_loss = np.mean(np.abs(gen_specs_mean - reference_mean)) + \
                     np.mean(np.abs(gen_specs_std - reference_std))
        
        # Calculate color distribution matching loss (helps match the blue-green appearance)
        gen_hist, _ = np.histogram(gen_specs.flatten(), bins=50, range=(-1, 1), density=True)
        color_dist_loss = np.mean(np.abs(gen_hist - real_hist))
        
        # Combined spectral loss
        spectral_loss = stats_loss + color_dist_loss
        spectral_losses.append(spectral_loss)
        
        # Train generator to fool discriminator
        g_loss = gan.train_on_batch(
            [noise, gen_conditions], 
            np.ones((batch_size, 1))
        )
        
        # Store losses
        d_losses.append(d_loss)
        g_losses.append(g_loss)
        
        # Print progress
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"[{machine_type}] Epoch {epoch+1}/{epochs} | "
                  f"D Loss: {d_loss:.4f} | G Loss: {g_loss:.4f} | "
                  f"Feature Loss: {feature_loss:.4f} | Spectral Loss: {spectral_loss:.4f}")
        
        # Save samples at specific intervals
        if (epoch + 1) in [1, 10, 50, 100, 200, 500, 1000, 2000, 3000] or (epoch + 1) % 250 == 0 or (epoch + 1) == epochs:
            save_generated_samples(generator, epoch + 1, machine_type, output_dir)
    
    # Save final model
    generator.save(f"{machine_type}_generator.h5")
    discriminator.save(f"{machine_type}_discriminator.h5")
    
    return d_losses, g_losses

def save_generated_samples(generator, epoch, machine_type, output_dir):
    """Save generated spectrograms with better comparison to originals"""
    # Generate noise and conditions
    n_samples = 3
    noise = np.random.normal(0, 1, (n_samples * 2, LATENT_DIM))
    
    # Create a batch with normal and abnormal conditions
    conditions = np.concatenate([
        np.zeros((n_samples, 1)),  # Normal
        np.ones((n_samples, 1))    # Abnormal
    ])
    
    # Generate spectrograms
    gen_specs = generator.predict([noise, conditions])
    
    # Create a grid of spectrograms
    plt.figure(figsize=(15, 10))
    
    for i in range(n_samples * 2):
        plt.subplot(2, n_samples, i+1)
        plt.imshow(gen_specs[i, :, :, 0], vmin=-1, vmax=1, cmap='viridis')
        condition_name = "Normal" if i < n_samples else "Abnormal"
        plt.title(f"{condition_name} (Epoch {epoch})")
        plt.colorbar()
        
        # Save individual spectrogram
        plt.figure(figsize=(8, 6))
        plt.imshow(gen_specs[i, :, :, 0], vmin=-1, vmax=1, cmap='viridis')
        plt.title(f"{condition_name} {machine_type.capitalize()} (Epoch {epoch})")
        plt.colorbar()
        plt.tight_layout()
        plt.savefig(f"{output_dir}/{condition_name.lower()}_sample_{i%n_samples+1}_epoch_{epoch}.png")
        plt.close()
        
        # Convert to audio and save
        audio = spectrogram_to_audio(gen_specs[i])
        sf.write(f"{output_dir}/{condition_name.lower()}_sample_{i%n_samples+1}_epoch_{epoch}.wav", audio, SR)
    
    # Close the main figure
    plt.tight_layout()
    plt.savefig(f"{output_dir}/samples_epoch_{epoch}.png")
    plt.close()

def generate_final_samples(generator, machine_type, output_dir, num_samples=200):
    """Generate final sample spectrograms and audio files"""
    
    final_dir = os.path.join(output_dir, "final_samples")
    os.makedirs(final_dir, exist_ok=True)
    
    for condition_label, condition_name in enumerate(["normal", "abnormal"]):
        condition_dir = os.path.join(final_dir, condition_name)
        os.makedirs(condition_dir, exist_ok=True)
        
        print(f"Generating {num_samples} {condition_name} samples...")
        for i in tqdm(range(num_samples)):
            # Generate noise
            noise = np.random.normal(0, 1, (1, LATENT_DIM))
            condition = np.array([[condition_label]])
            
            # Generate spectrogram
            gen_spec = generator.predict([noise, condition], verbose=0)
            
            # Save spectrogram as image
            plt.figure(figsize=(8, 6))
            plt.imshow(gen_spec[0, :, :, 0])
            plt.title(f"{condition_name.capitalize()} {machine_type.capitalize()} Sound")
            plt.colorbar()
            plt.tight_layout()
            plt.savefig(f"{condition_dir}/{condition_name}_sample_{i+1}.png")
            plt.close()
            
            # Convert to audio and save
            audio = spectrogram_to_audio(gen_spec[0])
            sf.write(f"{condition_dir}/{condition_name}_sample_{i+1}.wav", audio, SR)
    
    print(f"Final samples saved in '{final_dir}'")

# Function to create a summary report with example spectrograms
def create_summary_report(generator, machine_type, output_dir, d_losses, g_losses, conditions):
    """Create a summary report with example spectrograms"""
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    fig.suptitle(f'{machine_type.capitalize()} GAN Results', fontsize=16)
    
    # Plot loss curve
    axes[0, 0].plot(d_losses, label='Discriminator')
    axes[0, 0].plot(g_losses, label='Generator')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title('Training Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True, linestyle='--', alpha=0.7)
    
    # Get 2 normal and 2 abnormal examples
    normal_specs = []
    abnormal_specs = []
    
    # Generate 2 examples of each
    for _ in range(2):
        # Normal
        noise = np.random.normal(0, 1, (1, LATENT_DIM))
        normal_spec = generator.predict([noise, np.array([[0]])])
        normal_specs.append(normal_spec[0])
        
        # Abnormal
        noise = np.random.normal(0, 1, (1, LATENT_DIM))
        abnormal_spec = generator.predict([noise, np.array([[1]])])
        abnormal_specs.append(abnormal_spec[0])
    
    # Plot normal examples
    axes[0, 1].imshow(normal_specs[0][:, :, 0])
    axes[0, 1].set_title('Normal Example 1')
    axes[0, 2].imshow(normal_specs[1][:, :, 0])
    axes[0, 2].set_title('Normal Example 2')
    
    # Plot abnormal examples
    axes[1, 1].imshow(abnormal_specs[0][:, :, 0])
    axes[1, 1].set_title('Abnormal Example 1')
    axes[1, 2].imshow(abnormal_specs[1][:, :, 0])
    axes[1, 2].set_title('Abnormal Example 2')
    
    # Add dataset info
    axes[1, 0].axis('off')
    dataset_info = (
        f"Dataset Info:\n"
        f"Machine Type: {machine_type}\n"
        f"Normal Samples: {np.sum(conditions == 0)}\n"
        f"Abnormal Samples: {np.sum(conditions == 1)}\n"
        f"Training Epochs: {len(d_losses)}\n"
        f"Final D Loss: {d_losses[-1]:.4f}\n"
        f"Final G Loss: {g_losses[-1]:.4f}"
    )
    axes[1, 0].text(0.1, 0.5, dataset_info, fontsize=12, va='center')
    
    plt.tight_layout()
    plt.savefig(f"{output_dir}/summary_report.png", dpi=300)
    plt.close()
    
    print(f"Summary report saved as '{output_dir}/summary_report.png'")

# Run GAN training
# This is the fixed run_gan_training function to use
def run_gan_training(machine_type, data_dir, epochs=None, batch_size=None, samples_per_id=50):
    """Run GAN training for a specific machine type"""
    
    # Use global parameters if not specified
    if epochs is None:
        epochs = EPOCHS
    if batch_size is None:
        batch_size = BATCH_SIZE
        
    # Print selected configuration
    print(f"\n=== Training Configuration ===")
    print(f"Machine Type: {machine_type}")
    print(f"Data Directory: {data_dir}")
    print(f"Epochs: {epochs}")
    print(f"Batch Size: {batch_size}")
    print("============================\n")
    
    # Process data for the specified machine type
    machine_ids = ["00", "02", "04", "06"]
    specs, conditions = process_mimii_dataset(
        data_dir, 
        machine_type, 
        machine_ids, 
        samples_per_id
    )
    
    # Check if we have enough data
    if len(specs) == 0:
        print(f"Error: No data found for machine type {machine_type}")
        return None, None, None
    
    # Build models with explicit shapes
    generator = build_generator(LATENT_DIM)
    discriminator = build_discriminator()
    
    # Print model summaries
    print("\nGenerator Summary:")
    generator.summary()
    print("\nDiscriminator Summary:")
    discriminator.summary()
    
    # Test the generator to verify output shape
    noise = np.random.normal(0, 1, (1, LATENT_DIM))
    condition = np.array([[0]])  # Normal condition
    test_output = generator.predict([noise, condition])
    print(f"Generator test output shape: {test_output.shape}")
    
    # Test the discriminator to verify it accepts the generator's output
    disc_output = discriminator.predict([test_output, condition])
    print(f"Discriminator test output shape with generator output: {disc_output.shape}")
    
    # Build GAN
    gan = build_gan(generator, discriminator)
    
    # Compile models with lower learning rates
    discriminator.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.00005, beta_1=0.5),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    
    gan.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.00005, beta_1=0.5),
        loss='binary_crossentropy'
    )
    
    # Training with progress visualization
    try:
        # Our train_gan function now returns only d_losses and g_losses
        d_losses, g_losses = train_gan(generator, discriminator, gan, specs, conditions, epochs, batch_size, machine_type)
        
        # Create summary report
        output_dir = f"generated_samples_{machine_type}"
        create_summary_report(generator, machine_type, output_dir, d_losses, g_losses, conditions)
        
        print(f"Training complete!")
        return generator, discriminator, (d_losses, g_losses)
        
    except Exception as e:
        print(f"Error during training: {str(e)}")
        # Print detailed error info
        import traceback
        traceback.print_exc()
        return None, None, None

# Example usage in a notebook:

# Set your data directory
data_dir = "C:/Users/julia/DS4440/machine-noise-generator/data"

# Choose machine type: 'fan', 'pump', 'slider', or 'valve'
machine_type = "pump"  

# For quick testing, use fewer epochs
generator, discriminator, losses = run_gan_training(
    machine_type=machine_type,
    data_dir=data_dir,
    epochs=EPOCHS,
    samples_per_id=30
)

No normal fan spectrogram found. Using random data.
No abnormal fan spectrogram found. Using random data.
No normal pump spectrogram found. Using random data.
No abnormal pump spectrogram found. Using random data.
No normal valve spectrogram found. Using random data.
No abnormal valve spectrogram found. Using random data.
No normal slider spectrogram found. Using random data.
No abnormal slider spectrogram found. Using random data.


C:\Users\julia\AppData\Local\Temp\ipykernel_388\1139218588.py:131: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


Visualization saved to machine_spectrograms/all_machine_spectrograms.png
